In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
from scipy import stats

from astropy.time import Time
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
import astropy.units as u
from astropy.timeseries import TimeSeries
from astropy.coordinates import get_sun


from astroplan import Observer
from astroplan import FixedTarget
from astroplan.plots import plot_airmass, plot_parallactic
from astroplan import is_observable

from pytz import timezone

warnings.filterwarnings("ignore")
print(f"pandas   version : {pd.__version__}")
print(f"numpy    version : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

## Initialisation

### Target initialisations

In [ ]:
# ── LSST Deep Drilling Fields ─────────────────────────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

### Observatory initialisation

In [ ]:
# ── Rubin/LSST observatory location (Cerro Pachón) ───────────────────────────
RUBIN_LAT_DEG = -30.244728  # degrees North
RUBIN_LON_DEG = -70.749417  # degrees East  (West is negative)
RUBIN_HEIGHT_M = 2647.0  # metres above sea level

RUBIN_LOCATION = EarthLocation(
    lat=RUBIN_LAT_DEG * u.deg,
    lon=RUBIN_LON_DEG * u.deg,
    height=RUBIN_HEIGHT_M * u.m,
)
print(f"Rubin/LSST : lat={RUBIN_LAT_DEG}°  lon={RUBIN_LON_DEG}°  h={RUBIN_HEIGHT_M} m")

#### Observatory initialisation through astroplan

In [ ]:
# observer = Observer(name='Subaru Telescope',
#               location=location,
#               pressure=0.615 * u.bar,
#               relative_humidity=0.11,
#               temperature=0 * u.deg_C,
#               timezone=timezone('US/Hawaii'),
#               description="Subaru Telescope on Maunakea, Hawaii")

In [ ]:
observer = Observer.at_site("lsst")

In [ ]:
observer

## Start building the observation plan

#### Selected the field to be observed 

In [ ]:
selected_field_name = "COSMOS"

In [ ]:
coordinates = SkyCoord(
    DEEP_FIELDS[selected_field_name][0] * u.deg, DEEP_FIELDS[selected_field_name][1] * u.deg, frame="icrs"
)
field_target = FixedTarget(name=selected_field_name, coord=coordinates)

In [ ]:
field_target

### Observation times

In [ ]:
# 1. Définir début et fin
t_start = Time("2026-01-01 00:00:00")
t_end = Time("2026-06-30 23:59:59")

# 2. Construire grille temporelle (pas = 1 heure)
n_hours = int((t_end - t_start).to(u.hour).value)

times = t_start + np.arange(n_hours) * u.hour

## Plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

plot_airmass(
    field_target,
    observer,
    Time("2026-01-01"),
    ax=ax,
    brightness_shading=True,  # nuit/jour
    altitude_yaxis=True,  # altitude en plus
)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_parallactic(
    field_target,
    observer,
    Time("2026-01-01"),
    ax=ax,
    # brightness_shading=True,   # nuit/jour
    # altitude_yaxis=True        # altitude en plus
)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))


plot_airmass(field_target, observer, times, ax=ax, brightness_shading=True)

# Formatter personnalisé
ax.xaxis.set_major_formatter(mdates.DateFormatter("%y-%m-%d:%H"))

# Optionnel : espacer les ticks (sinon trop dense)
ax.xaxis.set_major_locator(mdates.AutoDateLocator())

# Rotation pour lisibilité
plt.xticks(rotation=45)

plt.tight_layout()

plt.show()

In [ ]:
# =========================
# 1. Time grid (6 mois, 1h)
# =========================
t_start = Time("2026-01-01 00:00:00")
t_end = Time("2026-06-30 23:00:00")

n_hours = int((t_end - t_start).to(u.hour).value)
times = t_start + np.arange(n_hours) * u.hour


# =========================
# 2. Figure setup
# =========================
fig, ax = plt.subplots(figsize=(14, 6), dpi=120)


# =========================
# 3. Plot airmass
# =========================
plot_airmass(
    field_target,
    observer,
    times,
    ax=ax,
    brightness_shading=True,  # nuit/jour
    altitude_yaxis=True,  # altitude à droite
)


# =========================
# 4. Contraintes Rubin
# =========================
# airmass < 1.5
airmass_max = 2.2
ax.axhline(airmass_max, ls="--", lw=2, color="red", alpha=0.7)
ax.text(times[0].datetime, airmass_max, f"Rubin limit (X={airmass_max:.1f})", color="red")


# =========================
# 5. Axe temps propre
# =========================
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%y-%m-%d"))

ax.xaxis.set_minor_locator(mdates.DayLocator(interval=1))

fig.autofmt_xdate()


# =========================
# 6. Labels
# =========================
ax.set_title("COSMOS visibility (Jan–Jun 2026)")
ax.set_ylabel("Airmass")


# =========================
# 7. Layout
# =========================
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 1. Time grid (6 mois, 1h)
# =========================
t_start = Time("2026-01-01 00:00:00")
t_end = Time("2026-06-30 23:00:00")

n_hours = int((t_end - t_start).to(u.hour).value)
times = t_start + np.arange(n_hours) * u.hour


# =========================
# 2. Coordonnées AltAz
# =========================
altaz_frame = observer.altaz(times, target=field_target)

alt = altaz_frame.alt
airmass = altaz_frame.secz


# =========================
# 3. Masque nuit (astronomique)
# =========================
sun_alt = observer.altaz(times, get_sun(times)).alt
night_mask = sun_alt < -18 * u.deg


# =========================
# 4. Préparer grille (date × heure)
# =========================
# heure locale
local_times = times.to_datetime(timezone=observer.timezone)

hours = np.array([t.hour + t.minute / 60 for t in local_times])
dates = np.array([t.date() for t in local_times])

unique_dates = np.unique(dates)
n_days = len(unique_dates)

# grille vide
grid = np.full((24, n_days), np.nan)

# remplissage
for i, (d, h, am, is_night) in enumerate(zip(dates, hours, airmass, night_mask)):
    if is_night and am > 0:  # visible + nuit
        day_idx = np.where(unique_dates == d)[0][0]
        hour_idx = int(h)
        grid[hour_idx, day_idx] = am

# directement les fenêtres exploitables
# grid[grid > airmass_max] = np.nan


# =========================
# 5. Plot heatmap
# =========================
fig, ax = plt.subplots(figsize=(10, 4))

im = ax.imshow(
    grid,
    origin="lower",
    aspect="auto",
    vmin=1,
    vmax=2,
)

# axe x = dates
ax.set_xticks(np.arange(0, n_days, 14))
ax.set_xticklabels([str(d) for d in unique_dates[::14]], rotation=45)

# axe y = heures
ax.set_yticks(np.arange(0, 24, 2))
ax.set_ylabel("Local hour")

# colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Airmass")

ax.set_title("COSMOS visibility heatmap (night only, Jan–Jun 2026)")

plt.tight_layout()
plt.show()